# Part 1 — Reproducing the Paper

**Question:** does Gemini 3.1 reproduce the qualitative ToT result on Mini Crosswords (paper, GPT-4)?  
**Configurations:** IO, CoT, ToT (unbatched), ToT + best state (oracle), ToT − prune, ToT − backtrack.  
**Data:** `results/runs/crosswords/*.jsonl` for the 20-puzzle paper split.

In [ ]:
import json, glob, statistics
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np

# Walk up from cwd to the repo root (where 'results/' and 'data/' live).
# Robust to Jupyter being launched from the notebook dir or the repo root.
ROOT = Path.cwd().resolve()
while not ((ROOT / 'results').is_dir() and (ROOT / 'data').is_dir()):
    if ROOT == ROOT.parent:
        raise RuntimeError(f"can't find repo root from {Path.cwd()}")
    ROOT = ROOT.parent
RUNS = ROOT / 'results' / 'crosswords' / 'runs'
assert RUNS.is_dir(), f'expected {RUNS} to exist'

# ---------------- Report palette ----------------
CHARCOAL = '#233d4d'   # primary dark
PUMPKIN  = '#fe7f2d'   # accent / highlight
GOLDEN   = '#fcca46'   # warm secondary
OLIVE    = '#a1c181'   # muted secondary
SEAGRASS = '#619b8a'   # cool secondary
PALETTE  = [CHARCOAL, PUMPKIN, GOLDEN, OLIVE, SEAGRASS]

# ---------------- Report-grade defaults ----------------
mpl.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.edgecolor':    CHARCOAL,
    'axes.labelcolor':   CHARCOAL,
    'axes.titlecolor':   CHARCOAL,
    'axes.titleweight':  'bold',
    'axes.labelweight':  'bold',
    'axes.titlesize':    14,
    'axes.labelsize':    12,
    'axes.linewidth':    1.6,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         False,
    'xtick.color':       CHARCOAL,
    'ytick.color':       CHARCOAL,
    'xtick.labelsize':   11,
    'ytick.labelsize':   11,
    'xtick.major.size':  0,
    'ytick.major.size':  4,
    'font.weight':       'bold',
    'font.family':       'sans-serif',
    'legend.frameon':    False,
    'legend.fontsize':   11,
    'lines.linewidth':   2.6,
    'lines.markersize':  9,
    'savefig.dpi':       180,
    'savefig.bbox':      'tight',
})

## 1. Helpers — same metrics as `analyze.py`

In [ ]:
def load_rows(pattern):
    rows = []
    for fp in sorted(glob.glob(str(RUNS / pattern))):
        with open(fp) as f:
            for line in f:
                if line.strip():
                    rows.append(json.loads(line))
    return rows

def grid_metrics(grid, gt):
    flat = [c.upper() for row in grid for c in row]
    truth = [c.upper() for c in gt]
    if len(flat) != 25:
        return 0.0, 0.0, 0.0
    correct_letters = sum(1 for o, t in zip(flat, truth) if o == t)
    correct_words = 0
    for i in range(5):
        if flat[i*5:(i+1)*5] == truth[i*5:(i+1)*5]: correct_words += 1
    for i in range(5):
        if flat[i::5] == truth[i::5]: correct_words += 1
    return correct_letters/25, correct_words/10, (1.0 if correct_letters == 25 else 0.0)

def best_state_from_trace(trace, gt):
    best = (0.0, 0.0, 0.0)
    for rec in trace or []:
        grid = [['_']*5 for _ in range(5)]
        for entry in rec.get('filled', []):
            if not entry or len(entry) < 2: continue
            cid, w = entry[0], (entry[1] or '').upper()
            if len(w) != 5 or len(cid) < 2: continue
            d, idx = cid[0], int(cid[1])-1
            if d == 'h':
                for c in range(5): grid[idx][c] = w[c]
            elif d == 'v':
                for r in range(5): grid[r][idx] = w[r]
        m = grid_metrics(grid, gt)
        if (m[1], m[0], m[2]) > (best[1], best[0], best[2]):
            best = m
    return best

def summarize(rows, label, use_best=False):
    if not rows: return None
    Ls, Ws, Gs, Ss, Cs = [], [], [], [], []
    for r in rows:
        if 'avg_letter_acc' in r:  # IO/CoT pre-averaged over 10 samples
            Ls.append(r['avg_letter_acc']); Ws.append(r['avg_word_acc']); Gs.append(r['avg_game_acc'])
            Ss.append(1.0); Cs.append(1.0)
        else:
            grid = r.get('grid', []); gt = r.get('ground_truth', [])
            l, w, g = (best_state_from_trace(r.get('trace', []), gt)
                       if use_best else grid_metrics(grid, gt))
            Ls.append(l); Ws.append(w); Gs.append(g)
            Ss.append(r.get('expansions', r.get('steps', 0)))
            Cs.append(r.get('llm_calls_total', 0))
    return {
        'label': label, 'n': len(rows),
        'letter': statistics.mean(Ls)*100, 'word': statistics.mean(Ws)*100, 'game': statistics.mean(Gs)*100,
        'steps': statistics.mean(Ss), 'calls': statistics.mean(Cs),
        'game_se': (statistics.pstdev(Gs)*100/(len(Gs)**0.5)) if len(Gs) > 1 else 0.0,
    }

## 2. Reload the Part-1 table

In [ ]:
ours = [
    summarize(load_rows('cw_gem31_io_papersplit*.jsonl'),  'IO'),
    summarize(load_rows('cw_gem31_cot_papersplit*.jsonl'), 'CoT'),
    summarize(load_rows('cw_gem31_tot_unbatched_s100_prune_backtrack_papersplit*.jsonl'), 'ToT'),
    summarize(load_rows('cw_gem31_tot_unbatched_s100_prune_backtrack_papersplit*.jsonl'), 'ToT + best', use_best=True),
    summarize(load_rows('cw_gem31_tot_unbatched_s100_noprune_backtrack_papersplit*.jsonl'), 'ToT − prune'),
    summarize(load_rows('cw_gem31_tot_unbatched_s100_prune_nobacktrack_papersplit*.jsonl'), 'ToT − backtrack'),
]
ours = [o for o in ours if o is not None]

paper = {
    'IO':              {'letter': 38.7, 'word': 14.0, 'game':  0.0},
    'CoT':             {'letter': 40.6, 'word': 15.6, 'game':  1.0},
    'ToT':             {'letter': 78.0, 'word': 60.0, 'game': 20.0},
    'ToT + best':      {'letter': 82.4, 'word': 67.5, 'game': 35.0},
    'ToT − prune':     {'letter': 65.4, 'word': 41.5, 'game':  5.0},
    'ToT − backtrack': {'letter': 54.6, 'word': 20.0, 'game':  5.0},
}

print(f'{"Method":18s}{"n":>4}{"L%":>8}{"W%":>8}{"G%":>8}{"steps":>8}{"calls":>9}')
print('-'*63)
for o in ours:
    print(f'{o["label"]:18s}{o["n"]:>4}{o["letter"]:>7.1f}%{o["word"]:>7.1f}%{o["game"]:>7.1f}%{o["steps"]:>8.1f}{o["calls"]:>9.1f}')

## 3. Plot 1 — Paper vs Ours, three metrics side-by-side

Paper = muted olive, Ours = charcoal blue. Bold value labels above bars.

In [ ]:
labels = [o['label'] for o in ours]
ours_metrics  = {m: [o[m] for o in ours] for m in ('letter', 'word', 'game')}
paper_metrics = {m: [paper[l][m] for l in labels] for m in ('letter', 'word', 'game')}

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
x = np.arange(len(labels))
w = 0.38
for ax, m, title in zip(axes, ('letter', 'word', 'game'),
                              ('Letter accuracy', 'Word accuracy', 'Game accuracy')):
    ax.bar(x - w/2, paper_metrics[m], w, color=OLIVE,    label='Paper (GPT-4)',     edgecolor='none')
    ax.bar(x + w/2, ours_metrics[m],  w, color=CHARCOAL, label='Ours (Gemini 3.1)', edgecolor='none')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=22, ha='right')
    ax.set_ylim(0, 100)
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.set_title(title)
    for xi, (pv, ov) in enumerate(zip(paper_metrics[m], ours_metrics[m])):
        ax.text(xi - w/2, pv + 2, f'{pv:.0f}', ha='center', fontsize=10, color=OLIVE,    fontweight='bold')
        ax.text(xi + w/2, ov + 2, f'{ov:.0f}', ha='center', fontsize=10, color=CHARCOAL, fontweight='bold')
axes[0].set_ylabel('%')
axes[0].legend(loc='upper left')
fig.suptitle('Paper (GPT-4) vs Ours (Gemini 3.1) — 20-puzzle paper split',
             fontsize=14, fontweight='bold', color=CHARCOAL, y=1.02)
fig.tight_layout()
plt.show()

## 4. Plot 2 — Ablations: game accuracy with cost overlay

Bars = game accuracy (charcoal). Line = mean API calls / puzzle (pumpkin, log right axis).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(ours))
ymax = max(o['game'] for o in ours)

bars = ax.bar(x, [o['game'] for o in ours], color=CHARCOAL, width=0.62, edgecolor='none')
for rect, o in zip(bars, ours):
    ax.text(rect.get_x() + rect.get_width()/2, rect.get_height() + ymax*0.025,
            f'{o["game"]:.1f}%', ha='center', va='bottom',
            fontsize=11, color=CHARCOAL, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=22, ha='right')
ax.set_ylabel('Game accuracy (%)', color=CHARCOAL)
ax.set_ylim(0, ymax * 1.4)
ax.tick_params(axis='y', colors=CHARCOAL)

# Cost line: only draw for the ToT variants — IO/CoT trivially cost 1.
ax2 = ax.twinx()
ax2.set_yscale('log')
ax2.spines['right'].set_visible(True)
ax2.spines['right'].set_color(PUMPKIN)
ax2.spines['top'].set_visible(False)

tot_idx   = [i for i, o in enumerate(ours) if o['label'] not in ('IO', 'CoT')]
tot_calls = [ours[i]['calls'] for i in tot_idx]
ax2.plot(tot_idx, tot_calls, color=PUMPKIN, marker='o',
         markeredgecolor='white', markeredgewidth=1.6, zorder=5)
for i, c in zip(tot_idx, tot_calls):
    ax2.annotate(f'{c:.0f}', (i, c), xytext=(0, 12), textcoords='offset points',
                 ha='center', fontsize=10, color=PUMPKIN, fontweight='bold')
ax2.set_xlim(ax.get_xlim())
ax2.set_ylabel('API calls / puzzle (log)', color=PUMPKIN)
ax2.tick_params(axis='y', colors=PUMPKIN)

ax.set_title('Game accuracy & cost across ToT ablations', pad=14)
fig.tight_layout()
plt.show()

## 5. Plot 3 — Cost / accuracy Pareto plane

Methods cycle through the palette; ToT (unbatched) is the lead in charcoal.

In [ ]:
method_color = {
    'IO':              GOLDEN,
    'CoT':             OLIVE,
    'ToT':             CHARCOAL,
    'ToT + best':      SEAGRASS,
    'ToT − prune':     PUMPKIN,
    'ToT − backtrack': '#c04a1c',
}
# Per-label annotation offset; nudges where two methods land at the same point.
label_offset = {
    'IO':              (12,  6),
    'CoT':             (12, -10),
    'ToT':             (12, -14),   # land below
    'ToT + best':      (12,  10),   # and above the same (cost, acc) point
    'ToT − prune':     (12,   4),
    'ToT − backtrack': (12,   4),
}

fig, ax = plt.subplots(figsize=(9, 6.2))
for o in ours:
    c = method_color.get(o['label'], CHARCOAL)
    ax.scatter(o['calls'], o['game'], s=240, color=c, edgecolor='white', linewidth=1.8, zorder=3)
    dx, dy = label_offset.get(o['label'], (12, 4))
    ax.annotate(o['label'], (o['calls'], o['game']),
                xytext=(dx, dy), textcoords='offset points',
                fontsize=11, color=c, fontweight='bold')

ax.set_xscale('log')
ax.set_xlabel('API calls / puzzle  (log scale)')
ax.set_ylabel('Game accuracy (%)')
ax.set_ylim(-3, max(o['game'] for o in ours) * 1.3)
ax.set_title('Cost vs accuracy — Part 1', pad=14)
fig.tight_layout()
plt.show()

## 6. Plot 4 — Per-puzzle solve overlap

Filled cell = puzzle solved by ≥1 run of that method. Color ramps white → charcoal.

In [ ]:
matrix = {}
io_rows  = load_rows('cw_gem31_io_papersplit*.jsonl')
cot_rows = load_rows('cw_gem31_cot_papersplit*.jsonl')
matrix['IO']  = {r['idx']: int(r.get('avg_game_acc', 0) > 0.5) for r in io_rows}
matrix['CoT'] = {r['idx']: int(r.get('avg_game_acc', 0) > 0.5) for r in cot_rows}

def solved_per_idx(rows):
    out = {}
    for r in rows:
        _, _, g = grid_metrics(r.get('grid', []), r.get('ground_truth', []))
        out[r['idx']] = max(out.get(r['idx'], 0), int(g > 0.5))
    return out

matrix['ToT']             = solved_per_idx(load_rows('cw_gem31_tot_unbatched_s100_prune_backtrack_papersplit*.jsonl'))
matrix['ToT − prune']     = solved_per_idx(load_rows('cw_gem31_tot_unbatched_s100_noprune_backtrack_papersplit*.jsonl'))
matrix['ToT − backtrack'] = solved_per_idx(load_rows('cw_gem31_tot_unbatched_s100_prune_nobacktrack_papersplit*.jsonl'))

all_idx = sorted({i for d in matrix.values() for i in d})
methods = list(matrix.keys())
M = np.array([[matrix[m].get(i, 0) for i in all_idx] for m in methods])

cmap = LinearSegmentedColormap.from_list('charcoal_fade', ['white', CHARCOAL])

fig, ax = plt.subplots(figsize=(13, 3.6))
ax.imshow(M, aspect='auto', cmap=cmap, vmin=0, vmax=1)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        if M[i, j]:
            ax.text(j, i, '●', ha='center', va='center', color='white', fontsize=11, fontweight='bold')
ax.set_yticks(range(len(methods)))
ax.set_yticklabels(methods)
ax.set_xticks(range(len(all_idx)))
ax.set_xticklabels([str(i) for i in all_idx], fontsize=10)
ax.set_xlabel('puzzle idx')
ax.tick_params(axis='x', length=0); ax.tick_params(axis='y', length=0)
for s in ('top', 'right', 'left', 'bottom'):
    ax.spines[s].set_visible(False)
ax.set_title('Puzzles solved by each method (any run)', pad=10)
fig.tight_layout()
plt.show()

io_solved  = {i for i, v in matrix['IO'].items() if v}
tot_solved = {i for i, v in matrix['ToT'].items() if v}
print(f'IO  solves: {len(io_solved):>2}  -> {sorted(io_solved)}')
print(f'ToT solves: {len(tot_solved):>2}  -> {sorted(tot_solved)}')
print(f'IO ∩ ToT:   {sorted(io_solved & tot_solved)}')
print(f'IO − ToT:   {sorted(io_solved - tot_solved)}')
print(f'ToT − IO:   {sorted(tot_solved - io_solved)}')

## 7. Insight check — `+ best state` oracle gain

Paper saw +15 pp on game accuracy. We expect ≈ 0.

In [ ]:
rows = load_rows('cw_gem31_tot_unbatched_s100_prune_backtrack_papersplit*.jsonl')
diffs = []
for r in rows:
    final = grid_metrics(r.get('grid', []), r.get('ground_truth', []))
    best  = best_state_from_trace(r.get('trace', []), r.get('ground_truth', []))
    diffs.append({
        'final_word': final[1], 'best_word': best[1],
        'final_game': final[2], 'best_game': best[2],
        'better': best[1] > final[1] + 1e-6,
    })
n = len(diffs); n_better = sum(d['better'] for d in diffs)
delta_word = sum(d['best_word']-d['final_word'] for d in diffs)/n*100
delta_game = sum(d['best_game']-d['final_game'] for d in diffs)/n*100
print(f'{n_better}/{n} ({n_better/n*100:.0f}%) runs had a better intermediate state')
print(f'mean Δ word (best − final): +{delta_word:.2f} pp')
print(f'mean Δ game (best − final): +{delta_game:.2f} pp')